In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # project root, so `src` imports work
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
from src import config

# 02 — Data Analysis Report (Deliverable B)

14 tasks, each answered with a number/table and a one- or two-sentence interpretation. Uses the
cleaned, joined master table from notebook 01. Writes `reports/B_analysis_report.md`.

**Method used throughout ("demand ratio").** For every observed zone-hour we compute an *expected*
demand = typical trips for that zone, weekday and hour × that zone's level in that week, fitted on
ordinary hours only (no event window, no holiday, no rain in the last 3 h). The weekly level removes the
growth trend, so `ratio = trips / expected = 1.25` means 25% above a normal hour of the same zone,
weekday and hour in the same week (`src/analysis.expected_demand`).

In [2]:
from src import analysis as A, features
from src.nbtools import Report
m = A.load_master()
events = A.load_events()
weather = A.load_weather()
zone_types = pd.read_csv(config.PROCESSED / "zone_types.csv", index_col=0)["zone_type"]
obs = m[m.row_status == "observed"].copy()
o = A.expected_demand(m)                       # ordinary = no events, no holiday, dry
o_any = A.expected_demand(m, exclude_rain=False)  # baseline that ignores rain (for B2.1)
R = Report("B — Data Analysis Report (team " + config.TEAM_NAME + ")")
R.p("Generated by `notebooks/02_analysis_report.ipynb` from `data/processed/master_train.csv`. "
    "Only observed zone-hours are used (outage, pre-launch and missing hours excluded). "
    "**Demand ratio** = trips / expected trips for the same zone, weekday and hour in the same week, "
    "fitted on ordinary hours (no event, holiday or rain) — see `src/analysis.py`.")
print(len(obs), o["ratio"].describe().round(2).to_dict())

83104 {'count': 83104.0, 'mean': 1.04, 'std': 0.37, 'min': 0.0, '25%': 0.82, '50%': 1.01, '75%': 1.22, 'max': 8.0}


## B1 — Demand patterns
### B1.1 Volume by zone

In [3]:
v = obs.groupby("zone").agg(total_trips=("trips", "sum"), zone_hours=("trips", "size"),
                            mean_per_zone_hour=("trips", "mean"), first_hour=("pickup_hour", "min"))
v["share_pct"] = 100 * v["total_trips"] / v["total_trips"].sum()
v["zone_type"] = zone_types
v = v.sort_values("total_trips", ascending=False)
v_since = obs[obs.pickup_hour >= "2025-03-15"].groupby("zone")["trips"].mean().rename("mean_since_15_mar")
v = v.join(v_since)
v["rank_by_mean_since_15_mar"] = v["mean_since_15_mar"].rank(ascending=False).astype(int)
top3 = v.index[:3].tolist()
R.h("B1 Demand patterns")
R.h("B1.1 Volume by zone", 3)
R.table(v.reset_index()[["zone", "zone_type", "total_trips", "share_pct", "zone_hours", "mean_per_zone_hour",
                         "mean_since_15_mar", "rank_by_mean_since_15_mar", "first_hour"]], floatfmt=".1f")
R.p(f"**{top3[0]}, {top3[1]} and {top3[2]}** carry the most demand ({v.share_pct[:3].sum():.1f}% of all trips between them, "
    f"{v.mean_per_zone_hour[:3].min():.0f}–{v.mean_per_zone_hour[:3].max():.0f} trips per zone-hour vs a city mean of {obs.trips.mean():.1f}). "
    f"Ayat only operated from {v.loc['Ayat','first_hour']:%d %b} so its total ({v.loc['Ayat','share_pct']:.1f}% share) understates it; "
    f"comparing on the common period since 15 Mar, Ayat averages {v.loc['Ayat','mean_since_15_mar']:.1f} trips/zone-hour "
    f"(rank {v.loc['Ayat','rank_by_mean_since_15_mar']} of 12).")
v

,total_trips,zone_hours,mean_per_zone_hour,first_hour,share_pct,zone_type,mean_since_15_mar,rank_by_mean_since_15_mar
zone,,,,,,,,
Merkato,285655.000,7050,40.518440,2025-01-01,12.083317,market,42.041145,1
Bole,277500.500,7075,39.222686,2025-01-01,11.738378,nightlife_airport,41.286127,2
Megenagna,270082.500,7063,38.239063,2025-01-01,11.424594,transport_hub,40.088576,3
Kazanchis,224383.500,7069,31.741901,2025-01-01,9.491509,business_district,33.197575,4
Piassa,199787.000,7074,28.242437,2025-01-01,8.451067,business_district,29.582464,5
Lideta,198216.000,7076,28.012436,2025-01-01,8.384613,transport_hub,29.411995,6
CMC,182395.000,7087,25.736560,2025-01-01,7.715379,residential,27.020391,7
Gerji,170607.000,7052,24.192711,2025-01-01,7.216742,residential,25.446832,8
Kolfe,162352.500,7073,22.953839,2025-01-01,6.867573,residential,24.144523,9


### B1.2 Hour-of-day profile by zone type

In [4]:
wd = obs[obs.is_weekend == 0]
prof = wd.groupby(["zone_type", "hour"])["trips"].mean().unstack(0)
shape = prof / prof.mean()
summ = pd.DataFrame({
    "zones": [", ".join(sorted(zone_types[zone_types == t].index)) for t in prof.columns],
    "morning_peak": [f"{shape[t].loc[5:11].idxmax()}:00 ({shape[t].loc[5:11].max():.1f}x)" for t in prof.columns],
    "evening_peak": [f"{shape[t].loc[12:23].idxmax()}:00 ({shape[t].loc[12:23].max():.1f}x)" for t in prof.columns],
    "overall_peak_hour": [int(shape[t].idxmax()) for t in prof.columns],
    "quietest_hour": [int(shape[t].idxmin()) for t in prof.columns],
    "night_share_0to5_pct": [100 * prof[t].loc[0:5].sum() / prof[t].sum() for t in prof.columns],
}, index=prof.columns)
R.h("B1.2 Hour-of-day profile by zone type (weekdays)", 3)
R.p("Zones were grouped by the shape of their weekday profile (each zone's hourly mean / its daily mean) "
    "plus the weekend ratio, with rules fitted on training data (`features.fit_zone_types`). Peaks are "
    "shown as multiples of the group's average hour.")
R.table(summ.reset_index(), floatfmt=".1f")
R.p(f"Business districts have a sharp commute double peak ({summ.loc['business_district','morning_peak']} and "
    f"{summ.loc['business_district','evening_peak']}) and are nearly empty at night; residential zones peak earlier in the "
    f"morning ({summ.loc['residential','morning_peak']}) as people leave home and again in the evening "
    f"({summ.loc['residential','evening_peak']}). Merkato (market) is a midday plateau that dies after 18:00, and "
    f"Bole (nightlife/airport) is the only zone with real late-night demand "
    f"({summ.loc['nightlife_airport','night_share_0to5_pct']:.0f}% of its trips between 00:00 and 05:59). Every group is quietest around 02:00–04:00.")
summ

,zones,morning_peak,evening_peak,overall_peak_hour,quietest_hour,night_share_0to5_pct
zone_type,,,,,,
business_district,"Arat Kilo, Kazanchis, Piassa",8:00 (2.0x),18:00 (2.6x),18,2,3.578679
market,Merkato,11:00 (2.0x),13:00 (2.1x),13,0,4.183006
nightlife_airport,Bole,6:00 (1.6x),20:00 (1.5x),6,2,15.217394
residential,"Ayat, CMC, Gerji, Kolfe, Sarbet",7:00 (2.1x),19:00 (2.0x),7,2,7.619219
transport_hub,"Lideta, Megenagna",8:00 (1.9x),18:00 (2.2x),18,1,6.223504


### B1.3 Weekday vs weekend

In [5]:
we = obs.groupby(["zone", "is_weekend"])["trips"].mean().unstack()
we.columns = ["weekday_mean", "weekend_mean"]
we["weekend_to_weekday"] = we["weekend_mean"] / we["weekday_mean"]
we["zone_type"] = zone_types
we = we.sort_values("weekend_to_weekday", ascending=False)
dow_bole = obs[obs.zone == "Bole"].groupby("dow")["trips"].mean()
dow_kaz = obs[obs.zone == "Kazanchis"].groupby("dow")["trips"].mean()
dow_tab = pd.DataFrame({"Bole": dow_bole, "Kazanchis": dow_kaz,
                        "Bole_index": dow_bole / dow_bole.mean(), "Kazanchis_index": dow_kaz / dow_kaz.mean()})
dow_tab.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
R.h("B1.3 Weekday vs weekend", 3)
R.table(we.reset_index(), floatfmt=".2f")
R.p("Day-of-week shape (mean trips per zone-hour) for the two extremes:")
R.table(dow_tab.reset_index().rename(columns={"index": "day"}), floatfmt=".2f")
busier = we.index[we.weekend_to_weekday > 1.05].tolist()
collapse = we.index[we.weekend_to_weekday < 0.6].tolist()
R.p(f"Zones busier at the weekend: {', '.join(busier)} — Bole most of all ({we.loc['Bole','weekend_to_weekday']:.2f}x), "
    f"with Saturday its busiest day ({dow_tab.loc['Sat','Bole_index']:.2f}x its average day). Weekend demand collapses to about half in the business "
    f"districts {', '.join(collapse)} ({we.loc[collapse,'weekend_to_weekday'].min():.2f}–{we.loc[collapse,'weekend_to_weekday'].max():.2f}x), "
    "so a model needs zone x weekday interactions, not a single weekend flag.")
we

,weekday_mean,weekend_mean,weekend_to_weekday,zone_type
zone,,,,
Bole,35.904428,47.549603,1.324338,nightlife_airport
Gerji,23.638553,25.585700,1.082372,residential
Sarbet,21.179806,22.895302,1.080997,residential
CMC,25.190791,27.109375,1.076162,residential
Kolfe,22.500296,24.097063,1.070966,residential
Ayat,17.562073,18.477880,1.052147,residential
Merkato,41.942948,36.926573,0.880400,market
Lideta,29.522530,24.222222,0.820466,transport_hub
Megenagna,40.379576,32.857960,0.813727,transport_hub


### B1.4 Trend

In [6]:
full_weeks = obs.assign(week=obs.pickup_hour.dt.to_period("W-SUN").dt.start_time)
older = full_weeks[full_weeks.zone != "Ayat"]
wk_hours = older.groupby("week")["trips"].size()
keep = wk_hours[wk_hours >= 0.95 * 11 * 168].index        # full weeks only (drops 1 Jan stub and the outage week)
nonayat = older.groupby("week")["trips"].sum().reindex(keep)
wk = full_weeks.groupby("week").agg(trips=("trips", "sum")).reindex(keep)
x = np.arange(len(keep))
slope, intercept = np.polyfit(x, nonayat.values, 1)
first4, last4 = nonayat.iloc[:4].mean(), nonayat.iloc[-4:].mean()
monthly = obs[obs.zone != "Ayat"].groupby(obs.pickup_hour.dt.month)["trips"].mean()
growth_month = (monthly.iloc[-1] / monthly.iloc[0] - 1) * 100
wk_out = pd.DataFrame({"week_start": wk.index, "city_trips_all_zones": wk.trips.values,
                       "city_trips_excl_ayat": nonayat.values})
R.h("B1.4 Trend", 3)
R.p(f"Weekly city-wide trips (full weeks only; Ayat excluded for a like-for-like series because it launched mid-March). "
    f"A straight line through the {len(wk)} full weeks gives **+{slope:,.0f} trips per week, every week**. "
    f"The first 4 full weeks averaged {first4:,.0f} trips/week and the last 4 {last4:,.0f}: **{(last4/first4-1)*100:.0f}% growth** "
    f"(mean trips per zone-hour {monthly.iloc[0]:.1f} in January vs {monthly.iloc[-1]:.1f} in October, +{growth_month:.0f}%).")
R.table(wk_out.iloc[np.r_[0:4, -4:0]], floatfmt=",.0f")
nov_proj = intercept + slope * (len(wk) + 1.5)
R.p(f"Implication: November should run above October — the trend line projects about {nov_proj:,.0f} trips/week for "
    f"the 11 older zones in early November vs {last4:,.0f} in the last 4 weeks. A model trained on the whole year without a "
    "level/trend feature would under-forecast, and a tree model cannot extrapolate a trend by itself, so we give it recent-level "
    "features (`lag_14d`, `zone_level_2to4w`) that already contain the growth.")
wk_out.tail()

,week_start,city_trips_all_zones,city_trips_excl_ayat
36,2025-09-22,67349.0,63694.5
37,2025-09-29,66206.0,62735.5
38,2025-10-06,62674.5,59243.0
39,2025-10-13,62526.5,59248.5
40,2025-10-20,63914.5,60442.5


## B2 — Weather
### B2.1 Time-zone check

In [7]:
w = weather.set_index("pickup_hour")
shift_rows = []
zh = o_any[(o_any.zone_type != "market") & o_any.hour.between(6, 22)]
for k in range(0, 7):          # k = hours added to the RAW weather clock (3 = UTC->EAT)
    rain_k = w["rain_mm"].copy()
    rain_k.index = rain_k.index - pd.Timedelta(hours=3) + pd.Timedelta(hours=k)
    r = rain_k.reindex(zh.pickup_hour).values
    ok = ~np.isnan(r)
    wet = r[ok] > 0
    shift_rows.append({"hours added to raw weather clock": k,
                       "misalignment vs. correct (h)": k - 3,
                       "corr(rain_mm, demand ratio)": np.corrcoef(r[ok], zh.ratio.values[ok])[0, 1],
                       "demand ratio, rainy hours": zh.ratio.values[ok][wet].mean(),
                       "demand ratio, dry hours": zh.ratio.values[ok][~wet].mean()})
shift_tab = pd.DataFrame(shift_rows)
shift_tab["rain uplift %"] = 100 * (shift_tab["demand ratio, rainy hours"] / shift_tab["demand ratio, dry hours"] - 1)
best = int(shift_tab.loc[shift_tab["corr(rain_mm, demand ratio)"].idxmax(), "hours added to raw weather clock"])
ev = __import__("src.cleaning", fromlist=["x"]).weather_clock_evidence(pd.read_csv(config.RAW / "weather_hourly.csv"))
R.h("B2 Weather")
R.h("B2.1 Time-zone check", 3)
R.p(f"Two independent pieces of evidence. (1) **Temperature:** on the raw clock the average temperature peaks at "
    f"{ev['temp_peak_hour_raw_iso_z']}:00 — a solar-noon peak that is too early for the afternoon maximum; after +3 h it peaks at "
    f"{ev['temp_peak_hour_raw_iso_z']+3}:00 EAT. The raw file also starts at {ev['first_row_raw']:%d %b %H:%M} and the forecast rows at "
    f"{ev['first_forecast_raw']:%d %b %H:%M}, which are exactly midnight EAT on 31 Dec and 1 Nov. (2) **Rain–demand alignment:** "
    "we shift the raw weather clock by 0–6 h and correlate rain with the demand ratio (non-market zones, 06:00–22:00).")
R.table(shift_tab, floatfmt=".3f")
R.p(f"The rain signal is strongest at +{best} h (UTC -> EAT): correlation {shift_tab['corr(rain_mm, demand ratio)'].max():.3f} and rainy hours "
    f"{shift_tab.loc[shift_tab['hours added to raw weather clock']==3, 'rain uplift %'].iloc[0]:.0f}% above dry ones. Joining on the raw clock "
    f"(0 h, i.e. trusting the timestamps as local) cuts the correlation to {shift_tab.loc[0,'corr(rain_mm, demand ratio)']:.3f} and the uplift to "
    f"{shift_tab.loc[0,'rain uplift %']:.0f}%: a 3-hour clock error would hide most of the weather signal.")
shift_tab

,hours added to raw weather clock,misalignment vs. correct (h),"corr(rain_mm, demand ratio)","demand ratio, rainy hours","demand ratio, dry hours",rain uplift %
0,0,-3,0.019220,1.039359,0.997500,4.196419
1,1,-2,0.029978,1.043863,0.996948,4.705915
2,2,-1,0.035335,1.049812,0.996177,5.384077
3,3,0,0.271096,1.180835,0.979485,20.556781
4,4,1,0.036498,1.049995,0.996396,5.379346
5,5,2,0.040337,1.045694,0.997233,4.859531
6,6,3,0.029348,1.043956,0.997863,4.619163


### B2.2 Rain effect by zone type

In [8]:
oa = o_any.copy()
oa["rainy"] = oa["rain_mm"] > 0
# Matched comparison: rainy hour vs mean of dry hours of the same zone, weekday and hour (same month).
oa["month"] = oa.pickup_hour.dt.month
dry_ref = (oa[~oa.rainy & (oa[A.EVENT_WINDOW_COLS].sum(axis=1) == 0) & (oa.is_public_holiday == 0)]
           .groupby(["zone", "month", "dow", "hour"])["trips"].mean().rename("dry_ref"))
rr = oa[oa.rainy & (oa[A.EVENT_WINDOW_COLS].sum(axis=1) == 0) & (oa.is_public_holiday == 0)].join(dry_ref, on=["zone", "month", "dow", "hour"]).dropna(subset=["dry_ref"])
rr["r"] = rr.trips / rr.dry_ref
b22 = rr.groupby("zone_type").agg(rainy_hours=("trips", "size"), rainy_mean_trips=("trips", "mean"),
                                  matched_dry_mean_trips=("dry_ref", "mean"), rain_vs_dry_ratio=("r", "mean"))
cis = rr.groupby("zone_type")["r"].apply(A.bootstrap_ci)
b22["ci95_low"] = [c[0] for c in cis.reindex(b22.index)]
b22["ci95_high"] = [c[1] for c in cis.reindex(b22.index)]
b22 = b22.sort_values("rain_vs_dry_ratio", ascending=False)
R.h("B2.2 Rain effect by zone type", 3)
R.p("Each rainy zone-hour (rain_mm > 0, no event/holiday) is compared with the mean of dry hours of the same zone, weekday "
    "and hour in the same month (a matched comparison that also removes the trend). `rain_vs_dry_ratio` is the mean of the "
    "per-hour ratios, with a bootstrap 95% CI.")
R.table(b22.reset_index(), floatfmt=".2f")
up = b22[b22.ci95_low > 1].index.tolist()
down = b22[b22.ci95_high < 1].index.tolist()
R.p(f"Rain raises demand in {', '.join(up)} (about +{(b22.loc[up,'rain_vs_dry_ratio'].min()-1)*100:.0f}% to "
    f"+{(b22.loc[up,'rain_vs_dry_ratio'].max()-1)*100:.0f}%), but **not everywhere**: in the market zone (Merkato) demand falls "
    f"{(1-b22.loc['market','rain_vs_dry_ratio'])*100:.0f}% because shoppers stay away from an open-air market. A single rain coefficient "
    "would be wrong; the model needs rain x zone-type interactions (a tree model learns them).")
b22

,rainy_hours,rainy_mean_trips,matched_dry_mean_trips,rain_vs_dry_ratio,ci95_low,ci95_high
zone_type,,,,,,
residential,2524,37.762332,29.920018,1.300203,1.285084,1.315605
nightlife_airport,422,52.199052,41.556675,1.292127,1.254907,1.328934
business_district,1467,52.840150,41.855658,1.285243,1.265906,1.306536
transport_hub,1030,59.918932,47.946278,1.275512,1.254186,1.297033
market,516,44.774225,56.561612,0.803173,0.777878,0.830379


### B2.3 Rain dose-response

In [9]:
labels = dict(enumerate(features.RAIN_LABELS))
rr2 = oa[(oa[A.EVENT_WINDOW_COLS].sum(axis=1) == 0) & (oa.is_public_holiday == 0)].join(dry_ref, on=["zone", "month", "dow", "hour"]).dropna(subset=["dry_ref"])
rr2["r"] = rr2["trips"] / rr2["dry_ref"]
dose = rr2.groupby(["zone_type", "rain_class"])["r"].mean().unstack().rename(columns=labels)
cnt = rr2.groupby("rain_class").size().rename(index=labels)
nm = rr2[rr2.zone_type != "market"]
per_mm = nm.groupby("rain_class").agg(mean_rain_mm=("rain_mm", "mean"), mean_ratio=("r", "mean")).rename(index=labels)
per_mm["uplift_pct"] = 100 * (per_mm.mean_ratio - 1)
per_mm["uplift_pct_per_mm"] = per_mm.uplift_pct / per_mm.mean_rain_mm.replace(0, np.nan)
R.h("B2.3 Rain dose-response", 3)
R.p("Rain classes: none (0 mm), light (0–2.5 mm/h), moderate (2.5–7.5), heavy (> 7.5). Cells are mean demand ratios vs the "
    "matched dry hours from B2.2 (so 'none' is ~1 by construction). Zone-hours per class: " +
    ", ".join(f"{k} {v:,}" for k, v in cnt.items()) + ".")
R.table(dose.reset_index(), floatfmt=".2f")
R.p("Uplift per millimetre of rain (all zone types except the market):")
R.table(per_mm.reset_index(), floatfmt=".2f")
R.p(f"Outside the market the response grows with intensity — light rain about +{per_mm.loc['light','uplift_pct']:.0f}%, moderate "
    f"+{per_mm.loc['moderate','uplift_pct']:.0f}%, heavy +{per_mm.loc['heavy','uplift_pct']:.0f}% — but **less than proportionally**: "
    f"each mm of rain adds about {per_mm.loc['light','uplift_pct_per_mm']:.0f}% in light rain, {per_mm.loc['moderate','uplift_pct_per_mm']:.0f}% in moderate and only "
    f"{per_mm.loc['heavy','uplift_pct_per_mm']:.0f}% in heavy rain, so the effect saturates above roughly 5 mm/h. Merkato falls further the harder it rains "
    f"({dose.loc['market','heavy']:.2f}x normal in heavy rain).")
dose

rain_class,none,light,moderate,heavy
zone_type,,,,
business_district,1.0,1.181628,1.379185,1.593893
market,1.0,0.878138,0.734591,0.584085
nightlife_airport,1.0,1.184898,1.373534,1.662559
residential,1.0,1.179848,1.387421,1.699562
transport_hub,1.0,1.170518,1.373465,1.568274


## B3 — Events & calendar
### B3.1 Public holidays

In [10]:
daily_mean = obs.groupby(obs.pickup_hour.dt.floor("D"))["trips"].mean()   # per zone-hour (robust to missing rows)
hol = events[(events.event_type == "public_holiday") & (events.status == "confirmed") & (events.start <= config.TRAIN_END)]
hol_days = set(hol.start.dt.floor("D"))
rows, zrows = [], []
zone_day = obs.groupby(["zone", obs.pickup_hour.dt.floor("D")])["trips"].mean()
for r in hol.sort_values("start").itertuples():
    d = r.start.floor("D")
    refs = [d + pd.Timedelta(weeks=k) for k in (-2, -1, 1, 2)]
    refs = [x for x in refs if x in daily_mean.index and x not in hol_days]
    idx = daily_mean[d] / daily_mean[refs].mean()
    rows.append({"holiday": r.event_name, "date": d.date(), "weekday": d.day_name()[:3],
                 "trips_per_zone_hour": daily_mean[d], "same_weekday_nearby_weeks": daily_mean[refs].mean(),
                 "index_vs_normal": idx, "reference_days": len(refs)})
    for z in config.ZONES:
        if (z, d) in zone_day.index:
            zr = [x for x in refs if (z, x) in zone_day.index]
            zrows.append({"holiday": r.event_name, "zone": z, "index": zone_day[(z, d)] / zone_day[[(z, x) for x in zr]].mean()})
b31 = pd.DataFrame(rows).sort_values("index_vs_normal")
b31["rank"] = range(1, len(b31) + 1)
zi = pd.DataFrame(zrows).pivot(index="holiday", columns="zone", values="index")
local = (zi.sub(b31.set_index("holiday")["index_vs_normal"], axis=0))
R.h("B3 Events & calendar")
R.h("B3.1 Public holidays", 3)
R.p("City-wide trips per zone-hour on each holiday vs the same weekday 1 and 2 weeks before and after "
    "(skipping other holidays). Index 0.80 = 20% below normal. Ranked from biggest drop.")
R.table(b31, floatfmt=".2f")
R.p("Zone index (zone's holiday demand vs its own normal same-weekday demand):")
R.table(zi.reset_index(), floatfmt=".2f")
n_down = int((b31.index_vs_normal < 0.95).sum()); n_up = int((b31.index_vs_normal > 1.05).sum())
big_local = local.stack().sort_values()
R.p(f"Not all holidays reduce demand: {n_down} of {len(b31)} lower it by more than 5% (deepest: {b31.iloc[0].holiday}, "
    f"{(1-b31.iloc[0].index_vs_normal)*100:.0f}% below normal) while {n_up} {'raises' if n_up == 1 else 'raise'} it ({b31.iloc[-1].holiday} {(b31.iloc[-1].index_vs_normal-1)*100:+.0f}%). "
    f"Reactions are local: the zone furthest below its holiday's city index is {big_local.index[0][1]} on {big_local.index[0][0]} "
    f"({big_local.iloc[0]:+.2f}), and the furthest above is {big_local.index[-1][1]} on {big_local.index[-1][0]} ({big_local.iloc[-1]:+.2f}) — "
    "business districts empty out on holidays while Bole and the residential zones hold up.")
b31

,holiday,date,weekday,trips_per_zone_hour,same_weekday_nearby_weeks,index_vs_normal,reference_days,rank
4,Good Friday,2025-04-18,Fri,21.608156,29.179538,0.740524,4,1
0,Genna (Ethiopian Christmas),2025-01-07,Tue,18.843137,24.633161,0.764950,2,2
9,Eid al-Adha,2025-06-06,Fri,24.228723,30.975658,0.782186,4,3
10,Mawlid,2025-09-04,Thu,26.776978,32.404494,0.826335,3,4
11,Enkutatash (Ethiopian New Year),2025-09-11,Thu,27.992806,33.369491,0.838874,3,5
3,Eid al-Fitr,2025-03-31,Mon,22.251773,26.413666,0.842434,4,6
8,Downfall of the Derg,2025-05-28,Wed,24.200000,28.691802,0.843446,4,7
6,International Labour Day,2025-05-01,Thu,23.952899,26.819451,0.893117,4,8
12,Meskel (Finding of the True Cross),2025-09-27,Sat,29.779570,31.191705,0.954727,4,9
7,Patriots' Victory Day,2025-05-05,Mon,22.053004,22.407805,0.984166,4,10


### B3.2 Event-window study: football

In [11]:
fb = events[(events.event_type == "football_match") & (events.status == "confirmed") & (events.start <= config.TRAIN_END)]
idx = o.set_index(["zone", "pickup_hour"])
rows = []
for r in fb.itertuples():
    s, e = r.start.floor("h"), (r.end - pd.Timedelta(seconds=1)).floor("h")
    for z in r.zones:
        for name, hrs in [("2 h before start", pd.date_range(s - pd.Timedelta(hours=2), s - pd.Timedelta(hours=1), freq="h")),
                          ("during the match", pd.date_range(s, e, freq="h")),
                          ("2 h after the end", pd.date_range(e + pd.Timedelta(hours=1), e + pd.Timedelta(hours=2), freq="h"))]:
            for h in hrs:
                if (z, h) in idx.index:
                    x = idx.loc[(z, h)]
                    rows.append({"window": name, "event_id": r.event_id, "trips": x.trips, "expected": x.expected})
fbw = pd.DataFrame(rows)
b32 = fbw.groupby("window", sort=False).agg(zone_hours=("trips", "size"), mean_trips=("trips", "mean"),
                                             same_zone_hours_without_event=("expected", "mean"))
b32["uplift_pct"] = 100 * (b32.mean_trips / b32.same_zone_hours_without_event - 1)
ci = fbw.assign(r=fbw.trips / fbw.expected).groupby("window", sort=False)["r"].apply(A.bootstrap_ci)
b32["ci95_pct"] = [f"{100*(a-1):+.0f} to {100*(b-1):+.0f}" for a, b in ci]
R.h("B3.2 Event-window study (football)", 3)
R.p(f"{len(fb)} confirmed matches before 1 Nov, all at Addis Ababa Stadium (Kazanchis). The comparison is the expected demand "
    "for the same zone, weekday and hour in the same week without any event.")
R.table(b32.reset_index(), floatfmt=".1f")
top = b32.uplift_pct.idxmax()
R.p(f"Every window is above normal, and **the 2 hours after the final whistle show the largest uplift ({b32.loc[top,'uplift_pct']:+.0f}%)** — "
    f"the whole crowd leaves at once — ahead of the arrival window ({b32.loc['2 h before start','uplift_pct']:+.0f}%). During the match the uplift is "
    f"smaller ({b32.loc['during the match','uplift_pct']:+.0f}%) because spectators are inside the stadium.")
b32

,zone_hours,mean_trips,same_zone_hours_without_event,uplift_pct,ci95_pct
window,,,,,
2 h before start,73,74.924658,41.249943,81.635784,+60 to +92
during the match,78,48.602564,34.783129,39.730282,+22 to +53
2 h after the end,77,64.285714,28.332873,126.894444,+104 to +160


### B3.3 Event-type ranking

In [12]:
rows = []
for t in ["football_match", "concert", "conference", "exhibition", "road_closure", "sports_run", "public_holiday", "school_break"]:
    e = events[(events.event_type == t) & (events.status == "confirmed") & (events.start <= config.TRAIN_END)]
    eh = __import__("src.cleaning", fromlist=["x"]).explode_event_hours(e)
    j = eh.merge(o_any[["zone", "pickup_hour", "trips", "expected"]], on=["zone", "pickup_hour"])
    r = j.trips / j.expected
    lo, hi = A.bootstrap_ci(r)
    rows.append({"event_type": t, "events": len(e), "zone_hours_in_window": len(j),
                 "window": "own dates" if t in ("public_holiday", "school_break", "road_closure") else
                           f"{config.EVENT_PRE_HOURS} h before to {config.EVENT_POST_HOURS} h after",
                 "effect_pct": 100 * (r.mean() - 1), "ci95_low_pct": 100 * (lo - 1), "ci95_high_pct": 100 * (hi - 1)})
b33 = pd.DataFrame(rows)
b33["measurable"] = np.where((b33.ci95_low_pct > 0) | (b33.ci95_high_pct < 0), "yes", "no (CI spans 0)")
b33 = b33.reindex(b33.effect_pct.abs().sort_values(ascending=False).index)
b33.insert(0, "rank", range(1, len(b33) + 1))
R.h("B3.3 Event-type ranking", 3)
R.p("Effect = mean demand ratio in the affected zone(s) during the event window, minus 1 (bootstrap 95% CI). The expected demand "
    "comes from ordinary hours, so the school-break row compares break weeks with the zone's ordinary weekly level.")
R.table(b33, floatfmt=".1f")
none = b33[b33.measurable != "yes"].event_type.tolist()
R.p(f"Concerts and football matches are the big movers ({b33.set_index('event_type').loc['concert','effect_pct']:+.0f}% and "
    f"{b33.set_index('event_type').loc['football_match','effect_pct']:+.0f}% over the whole window), and road closures are the one type that "
    f"**reduces** demand ({b33.set_index('event_type').loc['road_closure','effect_pct']:+.0f}%: cars cannot get in). "
    + (f"No measurable effect: {', '.join(none)}." if none else "Every type has a measurable effect.")
    + " Holidays only look neutral when pooled: B3.1 shows business districts and Merkato falling to about half while Bole and the "
    "residential zones rise, which cancels out in the average — so the model gets the holiday flag together with the zone.")
b33

,rank,event_type,events,zone_hours_in_window,window,effect_pct,ci95_low_pct,ci95_high_pct,measurable
0,1,football_match,41,268,2 h before to 3 h after,56.603467,46.650641,67.509987,yes
1,2,concert,25,218,2 h before to 3 h after,36.945475,30.491609,43.819746,yes
4,3,road_closure,21,337,own dates,-33.898831,-36.724184,-31.012321,yes
2,4,conference,35,472,2 h before to 3 h after,22.434222,18.421739,26.583747,yes
5,5,sports_run,2,19,2 h before to 3 h after,-12.488329,-26.628491,1.691863,no (CI spans 0)
6,6,public_holiday,13,3567,own dates,2.084996,-0.145737,4.427133,no (CI spans 0)
3,7,exhibition,7,667,2 h before to 3 h after,0.152880,-2.803968,3.669992,no (CI spans 0)
7,8,school_break,2,18805,own dates,0.117197,-0.339072,0.554948,no (CI spans 0)


### B3.4 Cancelled and unlisted events

In [13]:
canc = events[events.status == "cancelled"].copy()
conf = events[events.status == "confirmed"]
def twin(r):
    t = conf[(conf.event_type == r.event_type) & (conf.start == r.start) & conf.zones.map(lambda z: bool(set(z) & set(r.zones)))]
    return ", ".join(t.event_id)
canc["confirmed_duplicate_at_same_time"] = canc.apply(twin, axis=1)
rows = []
for r in canc.itertuples():
    es = A.event_study(o_any, canc[canc.event_id == r.event_id].assign(status="confirmed"), r.event_type,
                       offsets=range(-config.EVENT_PRE_HOURS, int((r.end - r.start).total_seconds() // 3600) + config.EVENT_POST_HOURS))
    rows.append({"event_id": r.event_id, "type": r.event_type, "zone": ",".join(r.zones), "start": r.start,
                 "confirmed twin": r.confirmed_duplicate_at_same_time or "-", "mean ratio in window": es["mean"].mean(),
                 "hours": int(es["count"].sum())})
b34a = pd.DataFrame(rows)
alone = b34a[b34a["confirmed twin"] == "-"]
same_type_conf = b33.set_index("event_type")
R.h("B3.4 Cancelled and unlisted events", 3)
R.p("**(a) Cancelled events.** Mean demand ratio across the would-be event window (1.00 = normal):")
R.table(b34a, floatfmt=".2f")
R.p(f"The {len(alone)} cancelled events without a confirmed twin average a ratio of {alone['mean ratio in window'].mean():.2f} — no footprint "
    f"(confirmed events of the same types average well above 1, see B3.3). The other {len(b34a)-len(alone)} 'cancelled' rows share zone, type "
    "and start time with a confirmed row, and demand did surge, so those events went ahead and the cancelled row is a stale duplicate. "
    "Decision: cancelled events are excluded from model features (kept as `ev_cancelled_window` for analysis only).")
# (b) unlisted spikes
oo = o.copy()
oo["excess"] = oo.trips - oo.expected
cand = oo[(oo.ratio >= 1.8) & (oo.excess >= 20) & (oo[A.EVENT_WINDOW_COLS].sum(axis=1) == 0) & (oo.is_public_holiday == 0)
          & (oo.ev_cancelled_window == 0) & (oo.rain_3h == 0)]
spikes = cand.sort_values("excess", ascending=False).head(8)[["zone", "pickup_hour", "trips", "expected", "ratio", "excess"]].copy()
spikes["weekday"] = spikes.pickup_hour.dt.day_name()
hyp = {
    ("Bole", "2025-04-12"): "Saturday night, Bole nightlife: likely an unlisted party/concert or a late flight bank at the airport",
    ("Merkato", "2025-05-17"): "Saturday morning market rush (a market/bazaar day not in the calendar)",
    ("Piassa", "2025-09-19"): "Friday morning in the old town: likely a procession/rally or government event in Arada",
    ("Merkato", "2025-01-06"): "Genna (Christmas) eve shopping: the day before the 7 Jan holiday",
    ("Gerji", "2025-06-23"): "Monday evening: possibly an unlisted residential-area gathering (wedding/funeral) or local transit disruption",
    ("Kolfe", "2025-09-26"): "Meskel eve (Demera bonfire day, 26 Sep) — the holiday table only lists 27 Sep",
    ("Bole", "2025-07-08"): "05:00 airport rush: an early flight bank (the airport is in Bole)",
    ("Kazanchis", "2025-05-15"): "midday government/UN meeting not in the calendar",
}
spikes["hypothesis"] = [hyp.get((z, f"{t:%Y-%m-%d}"), "unknown") for z, t in zip(spikes.zone, spikes.pickup_hour)]
R.p("**(b) Spikes with no listed event.** Dry zone-hours (no rain in the last 3 h) at least 1.8x expected and 20+ trips above it, "
    "outside every confirmed event window, holiday and cancelled-event window — the 8 largest by excess trips:")
R.table(spikes, floatfmt=".1f")
R.p(f"{len(cand)} such zone-hours exist in 10 months; the largest are isolated single hours. Several fall on the eve of a holiday "
    "(Genna eve, Meskel eve/Demera) or on weekend nights in Bole, suggesting the calendar misses holiday eves and nightlife events; "
    "they are a source of irreducible error for any model built on this calendar (see D7).")
spikes

,zone,pickup_hour,trips,expected,ratio,excess,weekday,hypothesis
17039,Bole,2025-04-12 23:00:00,208.0,100.132675,2.077244,107.867325,Saturday,"Saturday night, Bole nightlife: likely an unli..."
68937,Merkato,2025-05-17 09:00:00,152.0,80.834686,1.880381,71.165314,Saturday,Saturday morning market rush (a market/bazaar ...
79232,Piassa,2025-09-19 08:00:00,124.0,66.397620,1.867537,57.602380,Friday,Friday morning in the old town: likely a proce...
65802,Merkato,2025-01-06 18:00:00,75.0,32.271022,2.324066,42.728978,Monday,Genna (Christmas) eve shopping: the day before...
33355,Gerji,2025-06-23 19:00:00,85.0,43.550049,1.951777,41.449951,Monday,Monday evening: possibly an unlisted residenti...
50215,Kolfe,2025-09-26 07:00:00,89.0,48.013159,1.853659,40.986841,Friday,"Meskel eve (Demera bonfire day, 26 Sep) — the ..."
19109,Bole,2025-07-08 05:00:00,88.0,47.261732,1.861972,40.738268,Tuesday,05:00 airport rush: an early flight bank (the ...
39708,Kazanchis,2025-05-15 12:00:00,52.0,25.495307,2.039591,26.504693,Thursday,midday government/UN meeting not in the calendar


## B4 — Operations & data quality
### B4.1 Operational variables vs demand

In [14]:
ops = obs[["trips", "active_drivers", "avg_wait_min", "avg_fare_birr"]]
corr = ops.corr()["trips"].drop("trips").to_frame("pearson_r_with_trips")
corr["spearman_rho"] = ops.corr(method="spearman")["trips"].drop("trips")
corr["trips_per_driver_median"] = [ (obs.trips / obs.active_drivers).median(), np.nan, np.nan]
R.h("B4 Operations & data quality")
R.h("B4.1 Operational variables vs demand", 3)
R.table(corr.reset_index().rename(columns={"index": "variable"}), floatfmt=".3f")
R.p(f"`active_drivers` moves almost one-for-one with trips (r = {corr.loc['active_drivers','pearson_r_with_trips']:.2f}, about "
    f"{(obs.trips/obs.active_drivers).median():.2f} trips per driver-hour): drivers go where demand is, so it is a *consequence* of demand, "
    f"not a cause. `avg_wait_min` rises with demand (r = {corr.loc['avg_wait_min','pearson_r_with_trips']:.2f}) because busy hours stretch supply, "
    f"and `avg_fare_birr` is unrelated to volume (r = {corr.loc['avg_fare_birr','pearson_r_with_trips']:.2f}) — it differs by zone (trip length), not by hour. "
    "None of the three exists for next Tuesday at 18:00, so none can be a forecast input; D4 shows how badly a model that used "
    "`active_drivers` would be fooled. We use them for analysis and for the demo (drivers needed = trips / 1.3, fares = trips x zone average fare).")
corr

,pearson_r_with_trips,spearman_rho,trips_per_driver_median
active_drivers,0.986862,0.992630,1.307692
avg_wait_min,0.533573,0.639338,NaN
avg_fare_birr,0.005525,0.072019,NaN


### B4.2 Gaps and outages

In [15]:
st = m[m.row_status.isin(["outage", "prelaunch", "random_gap", "missing_value"])]
def runs(df):
    out = []
    for z, g in df.sort_values("pickup_hour").groupby("zone"):
        blk = (g.pickup_hour.diff() != pd.Timedelta(hours=1)).cumsum()
        for _, b in g.groupby(blk):
            out.append({"zone": z, "from": b.pickup_hour.min(), "to": b.pickup_hour.max(), "hours": len(b)})
    return pd.DataFrame(out)
outage = st[st.row_status == "outage"]
pre = st[st.row_status == "prelaunch"]
rg = st[st.row_status == "random_gap"]
mv = st[st.row_status == "missing_value"]
periods = pd.DataFrame([
    {"type": "platform outage (all zones)", "from": outage.pickup_hour.min(), "to": outage.pickup_hour.max(),
     "hours": outage.pickup_hour.nunique(), "zone_hours": len(outage), "treatment": "excluded from training; lags falling in it are NaN"},
    {"type": "zone not launched (Ayat)", "from": pre.pickup_hour.min(), "to": pre.pickup_hour.max(),
     "hours": pre.pickup_hour.nunique(), "zone_hours": len(pre), "treatment": "excluded; not zero demand"},
    {"type": "random missing records", "from": rg.pickup_hour.min(), "to": rg.pickup_hour.max(),
     "hours": rg.pickup_hour.nunique(), "zone_hours": len(rg), "treatment": "excluded from training; LightGBM handles NaN lags"},
    {"type": "record present, trips blank or -1", "from": mv.pickup_hour.min(), "to": mv.pickup_hour.max(),
     "hours": mv.pickup_hour.nunique(), "zone_hours": len(mv), "treatment": "excluded from training and scoring"},
])
rg_runs = runs(rg)
city_hours_no_rows = int((m.groupby("pickup_hour")["row_status"].apply(lambda s: s.isin(["outage", "prelaunch", "random_gap"]).all())).sum())
R.h("B4.2 Gaps and outages", 3)
R.p(f"Every hour of 1 Jan–31 Oct for which a zone has no usable trips value. {city_hours_no_rows} hours have no rows in any zone, all of them "
    "inside the outage.")
R.table(periods)
R.p(f"Random gaps per zone ({len(rg)} zone-hours in {len(rg_runs)} runs; longest run {rg_runs.hours.max()} h):")
R.table(rg.groupby("zone").size().rename("random_gap_hours").to_frame().T, index=False)
R.p("Separation rule: an hour with no rows in every launched zone is an **outage** (42 consecutive hours on 12–13 May — the platform was down); "
    "hours before a zone's first record are **pre-launch** (only Ayat, until 15 Mar); everything else is **random missing**, which is "
    "scattered (runs of 1–2 hours, spread evenly over zones and months). None is filled with zero; all are excluded from training and scoring.")
periods

,type,from,to,hours,zone_hours,treatment
0,platform outage (all zones),2025-05-12 02:00:00,2025-05-13 19:00:00,42,504,excluded from training; lags falling in it are...
1,zone not launched (Ayat),2025-01-01 00:00:00,2025-03-14 23:00:00,1752,1752,excluded; not zero demand
2,random missing records,2025-01-01 08:00:00,2025-10-31 16:00:00,656,682,excluded from training; LightGBM handles NaN lags
3,"record present, trips blank or -1",2025-01-01 08:00:00,2025-10-31 19:00:00,1375,1510,excluded from training and scoring


### B4.3 Pay-period effect

In [16]:
hol_days = set(events.loc[(events.event_type == "public_holiday") & (events.status == "confirmed"), "start"].dt.floor("D"))
dly = obs[obs.zone != "Ayat"].groupby(obs.pickup_hour.dt.floor("D"))["trips"].mean().rename("trips_per_zone_hour").to_frame()
dly = dly[~dly.index.isin(hol_days)]
dly = dly[dly.index.to_series().between("2025-01-02", "2025-10-31")]
# Equal detrending for every day: centred 29-day rolling median, then the weekday factor.
dly["trend"] = dly.trips_per_zone_hour.rolling(29, center=True, min_periods=15).median()
dly["detr"] = dly.trips_per_zone_hour / dly.trend
dow_f = dly.groupby(dly.index.dayofweek)["detr"].mean()
dly["detr"] = dly["detr"] / dly.index.dayofweek.map(dow_f).values
dly["dom"] = dly.index.day
dly["payday"] = ((dly.index.day > dly.index.days_in_month - 5) | (dly.index.day <= 2)).astype(int)
dom_ratio = dly.groupby("dom")["detr"].mean()
pay = dly.groupby("payday")["detr"].agg(["mean", "size"])
pay.index = ["ordinary days", "payday window (last 5 + first 2 days)"]
eff = (pay.iloc[1]["mean"] / pay.iloc[0]["mean"] - 1) * 100
rng = np.random.default_rng(config.RANDOM_STATE)
a, b = dly.loc[dly.payday == 1, "detr"].values, dly.loc[dly.payday == 0, "detr"].values
boot = [rng.choice(a, len(a)).mean() / rng.choice(b, len(b)).mean() - 1 for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5]) * 100
R.h("B4.3 Pay-period effect", 3)
R.p("City-wide trips per zone-hour per day (11 zones open all year, holidays removed), each day divided by a centred 29-day rolling "
    "median (the trend) and by its weekday factor — so growth and weekday mix are removed the same way from every day. Mean by day of month:")
R.table(dom_ratio.to_frame("detrended_index").T, floatfmt=".3f")
R.table(pay.reset_index().rename(columns={"index": "days"}), floatfmt=".3f")
R.p(f"Days in the payday window run **{eff:+.1f}%** above equally-detrended ordinary days (bootstrap 95% CI {lo:+.1f}% to {hi:+.1f}%). "
    + ("The effect is small but clearly non-zero and known in advance from the calendar, so we keep `is_payday_window` as a feature "
       "(D5 and fig12 show whether the model uses it)." if lo > 0 else
       "The interval includes zero, so the effect is too small to rely on; we keep the flag only because it is free and let the "
       "importance analysis (fig12) decide."))
pay

,mean,size
ordinary days,0.984652,226
payday window (last 5 + first 2 days),1.054199,64


## Supplementary — Feature correlation matrix

In [17]:
import matplotlib.pyplot as plt
from src import plotstyle as S
S.apply()
CORR_COLS = {"trips": "trips (target)", "profile_zone_dow_hour": "zone x weekday x hour profile",
             "lag_mean_2to4w": "mean lag 2-4 weeks", "lag_14d": "lag 14 days", "lag_28d": "lag 28 days",
             "zone_level_2to4w": "zone level 2-4 weeks", "trend_days": "trend (days)", "hour": "hour",
             "is_weekend": "weekend", "is_public_holiday": "public holiday", "is_holiday_eve": "holiday eve",
             "is_payday_window": "payday window", "temp_c": "temperature", "humidity_pct": "humidity",
             "wind_kmh": "wind", "rain_mm": "rain (mm)", "rain_3h": "rain, last 3 h", "ev_during": "event under way",
             "ev_log_attendance": "event attendance (log)"}
corr_m = obs[list(CORR_COLS)].corr(method="spearman").rename(index=CORR_COLS, columns=CORR_COLS)
mask = np.triu(np.ones(corr_m.shape, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(np.ma.masked_array(corr_m.values, mask), cmap="RdBu_r", vmin=-1, vmax=1)
n = len(corr_m)
for i in range(n):
    for j in range(i + 1):
        v = corr_m.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if abs(v) > 0.6 else S.INK)
ax.set_xticks(range(n), corr_m.columns, rotation=50, ha="right"); ax.set_yticks(range(n), corr_m.index)
ax.grid(False); ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman correlation")
ax.set_title("Spearman correlation of forecast-time features with each other and with trips (observed zone-hours, Jan-Oct 2025)",
             fontsize=12, fontweight="bold")
fig.savefig(config.REPORTS / "b_correlation_matrix.png", dpi=150, bbox_inches="tight"); plt.close(fig);

with_trips = corr_m["trips (target)"].drop("trips (target)").sort_values(key=abs, ascending=False)
pairs = corr_m.where(np.tril(np.ones(corr_m.shape, dtype=bool), k=-1)).stack()
pairs = pairs[[a != "trips (target)" and b != "trips (target)" for a, b in pairs.index]]
top_pairs = pairs.abs().sort_values(ascending=False).head(6)
top_pairs = pd.DataFrame({"feature_a": [a for a, _ in top_pairs.index], "feature_b": [b for _, b in top_pairs.index],
                          "spearman_rho": pairs[top_pairs.index].values})
R.h("Supplementary: feature correlation matrix")
R.p("Spearman correlation (rank-based, so it also catches curved relationships) between the forecast-time features and trips, "
    f"over all {len(obs):,} observed zone-hours. The operational columns (fares, wait times, active drivers) are left out "
    "because they are not known at forecast time (see B4.1).")
R.img("b_correlation_matrix.png", "Feature correlation matrix")
R.p("**Correlation with trips, strongest first**")
R.table(with_trips.to_frame("spearman_rho").reset_index().rename(columns={"index": "feature"}), floatfmt=".3f")
R.p("**Most strongly related feature pairs**"); R.table(top_pairs, floatfmt=".3f")
r_t = corr_m.loc["temperature", "trips (target)"]; r_th = corr_m.loc["temperature", "hour"]
r_rain = corr_m.loc["rain (mm)", "trips (target)"]
R.p(f"The history features carry most of the signal: the zone x weekday x hour profile alone has rho = "
    f"{corr_m.loc['zone x weekday x hour profile', 'trips (target)']:.2f} with trips, and the lag features are almost copies of "
    f"each other (rho up to {top_pairs.spearman_rho.abs().max():.2f}), so each extra lag adds little on its own. Raw weather "
    f"correlations are confounded by the time of day: temperature looks strongly positive (rho = {r_t:.2f}) mainly because it peaks "
    f"in the daytime when demand is high (temperature vs hour rho = {r_th:.2f}), and the rain correlation (rho = {r_rain:.2f}) mixes "
    "rain's real effect with the fact that it falls in the busy afternoon. That is why B2 measures weather and events against the "
    "expected demand for the same zone, weekday and hour instead of reading this matrix directly.")
with_trips.round(3)

zone x weekday x hour profile    0.935
mean lag 2-4 weeks               0.925
lag 14 days                      0.897
lag 28 days                      0.895
temperature                      0.501
hour                             0.407
humidity                        -0.369
zone level 2-4 weeks             0.272
rain, last 3 h                   0.244
rain (mm)                        0.237
trend (days)                     0.115
event attendance (log)           0.107
event under way                  0.081
weekend                         -0.065
public holiday                  -0.033
payday window                    0.032
wind                            -0.010
holiday eve                     -0.004
Name: trips (target), dtype: float64

In [18]:
R.write(config.REPORTS / "B_analysis_report.md")

wrote B_analysis_report.md (35,407 chars)
